# Roshni v2: more cameras in training (pre-registered)

This run follows `PREREGISTRATION_v2.md`, committed to GitHub **before** this notebook was run. It is the v1 pipeline with one change: the OIA-DDR photos (147 hospitals, 42 camera models) are added to training and validation. APTOS test and DeepDRiD are scored once, after the cut-off is frozen.

### Before you run (Kaggle)
1. *Add Input*: **APTOS 2019 Blindness Detection** (competition) and the dataset **ddrdataset** (by mariaherrerot).
2. *Settings*: Accelerator **GPU**, Internet **On**.
3. **Save Version → Save & Run All (Commit)**. About 1 to 1.5 hours. Then download `roshni_v2_outputs.zip` from the version's Output.

> This project is a prototype and is not medical advice.

## The pre-registration (as committed)

# Pre-registration: Roshni v2, more cameras in training

**Written and committed before the v2 model was trained.** The Git commit that adds this file is the timestamp. Nothing below will be changed after v2's results are seen. Results that come out differently from what we hoped are reported in full.

## Why

The v1 model (trained on APTOS 2019 only, one Indian provider) ranked the unseen DeepDRiD hospital photos well (AUC 0.911). But at its frozen cut-off it caught only 56.1% of referable eyes there. Our reading: the cut-off learned on one source's cameras does not carry over to new cameras.

**Question:** does adding photos from many hospitals and camera models to training and validation make the frozen cut-off carry over better to hospitals the model has never seen?

## What changes (and nothing else)

- **Added data:** the OIA-DDR grading photos (Li et al. 2019): 13,673 photos from 147 hospitals in 23 provinces of China, taken with 42 camera models, graded 0 to 4 plus "ungradable". Licence: CC BY-NC-SA 4.0. Ungradable photos are excluded from grading.
- **Split:** OIA-DDR is split exactly as APTOS was. Near-duplicate photos (256-bit difference hash, at most 10 bits apart) are grouped. Then 7 stratified group folds with seed 0 give one test fold, one validation fold and five training folds. The APTOS split is the same as in v1.
- **Training:** APTOS training split plus OIA-DDR training split.
- **Validation:** APTOS validation plus OIA-DDR validation. The checkpoint (best validation AUC) and the referral cut-off (90% sensitivity) are chosen on this combined set, then frozen.
- **Unchanged:**
  - Model and training: EfficientNet-B0 with ImageNet weights, 384 px, 12 epochs, learning rate 3e-4, batch size 32, same augmentation, seed 0.
  - Targets: 90% sensitivity target, and the quality cut-off of 0.5.
  - Quality check: retrained the same way on DeepDRiD training-photo quality labels.
  - The decision rule.

## Tests (each scored once, after freezing)

1. **Primary: DeepDRiD (all 1,600 graded photos, 400 patients).** These hospitals were never used to train or to set the cut-off.
2. The 400-photo DeepDRiD validation workflow (refer / retake / rescreen, with the quality check).
3. The same 352 APTOS test photos as v1.
4. The OIA-DDR test split. This is new photos from the hospitals in training, so it is reported separately and is not external.
5. The same stress test as v1 (4 changes × 2 strengths).

**Safeguard:** every DeepDRiD photo is hash-checked against the OIA-DDR photos used for training and validation. If any near-copies exist, the primary result is also reported with them removed.

## Outcomes

- **Primary:** sensitivity and specificity on DeepDRiD at v2's frozen cut-off.
- **Secondary:**
  - DeepDRiD AUC;
  - referable eyes given an all-clear in the 400-photo workflow;
  - APTOS test AUC;
  - the stress test;
  - the same post-hoc local re-calibration analysis as v1 (`local_calibration.py`).

## Decision rule for the demo and the headline (fixed now)

v2 replaces v1 in the browser demo and the README headline **only if all five hold**:

1. DeepDRiD sensitivity at the frozen cut-off is **at least 66.1%** (v1's 56.1% plus 10 points).
2. DeepDRiD specificity is **at least 85.0%**.
3. APTOS test AUC is **at least 0.950**.
4. Referable eyes given an all-clear in the 400-photo workflow, with the quality check, are **at most 46** (v1's number).
5. Fewer than 1% of DeepDRiD photos (fewer than 16 of 1,600) have a near-copy among the OIA-DDR photos used for training or validation. Otherwise DeepDRiD is no longer unseen, so v2 cannot be promoted, whatever its numbers.

If any of these fails, v1 stays the headline and v2 is reported in full as a tested idea that did not meet its bar. Either way, both models' numbers are published side by side.

## What would not count as evidence

- **The OIA-DDR test result.** It is the same hospitals as training.
- **Any threshold chosen after looking at DeepDRiD.**
- **One more caution:** both OIA-DDR and DeepDRiD come from China. An improvement on DeepDRiD would show the value of camera variety, not that the model works in Pakistan. Testing on Pakistani clinic photos remains the next step.


## 1. Code (the same files as the GitHub repository)

In [ ]:
%%writefile drscreen.py
"""Roshni: diabetic-eye screening for a primary-care diabetes clinic in Pakistan.

One retinal photo in, one of three next steps out:
  REFER     - referable diabetic retinopathy suspected (moderate NPDR or worse)
  RETAKE    - photo not good enough to give an all-clear; take it again
  RESCREEN  - no referable disease seen on a good photo; screen again in 12 months

Safety rules built into the code:
  * A poor photo can never produce an all-clear, but it never blocks a referral either.
  * The referral threshold is chosen on the validation split to reach a sensitivity target
    fixed in advance (90%), then frozen before any test image is scored.
  * Near-duplicate photos are grouped before splitting, so a copy of a test image can never
    sit in training (a common way to get results that are too good to be true).
  * The external test set (DeepDRiD, China) is never used for training or for any threshold.

Prototype for a hackathon. Not medical advice and not a diagnosis.
"""
from __future__ import annotations

import os
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image

GRADES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]
REFERABLE_FROM = 2           # moderate NPDR or worse = "more-than-mild DR", the usual referral line
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


# ============================================================ preprocessing
def crop_fundus(img: Image.Image, tol: int = 10) -> Image.Image:
    """Cut away the black border around the round fundus image."""
    a = np.asarray(img.convert("L"))
    mask = a > tol
    if mask.sum() < 0.05 * a.size:          # almost all dark: leave it (the quality gate will judge it)
        return img
    ys, xs = np.where(mask)
    return img.crop((xs.min(), ys.min(), xs.max() + 1, ys.max() + 1))


def prep_image(path_or_img, size: int = 384) -> Image.Image:
    img = path_or_img if isinstance(path_or_img, Image.Image) else Image.open(path_or_img)
    img = crop_fundus(img.convert("RGB"))
    w, h = img.size
    side = max(w, h)
    canvas = Image.new("RGB", (side, side), (0, 0, 0))
    canvas.paste(img, ((side - w) // 2, (side - h) // 2))
    return canvas.resize((size, size), Image.BILINEAR)


def _cache_one(args):
    src, dst, size = args
    if not os.path.exists(dst):
        try:
            prep_image(src, size).save(dst, quality=95)
        except Exception as e:                # a broken file should not stop the run
            return f"FAILED {src}: {e}"
    return None


def cache_images(srcs, out_dir, size=384, workers=4):
    """Crop, pad and resize every image once, so training reads small JPEGs."""
    from multiprocessing import Pool
    os.makedirs(out_dir, exist_ok=True)
    dsts = [os.path.join(out_dir, os.path.splitext(os.path.basename(s))[0] + ".jpg") for s in srcs]
    jobs = [(s, d, size) for s, d in zip(srcs, dsts)]
    t0 = time.time()
    with Pool(workers) as pool:
        errs = [e for e in pool.imap(_cache_one, jobs, chunksize=8) if e]
    print(f"Cached {len(dsts)} images to {out_dir} in {time.time() - t0:.0f}s; {len(errs)} failed")
    for e in errs[:5]:
        print(" ", e)
    return dsts


# ============================================================ near-duplicates
def dhash(path, n: int = 16) -> np.ndarray:
    a = np.asarray(Image.open(path).convert("L").resize((n + 1, n), Image.BILINEAR), dtype=np.float32)
    return (a[:, 1:] > a[:, :-1]).ravel()


def duplicate_groups(paths, max_dist: int = 10):
    """Group photos whose 256-bit difference hashes are within `max_dist` bits.
    Returns (group id per image, list of near-duplicate pairs)."""
    H = np.stack([dhash(p) for p in paths]).astype(np.float32)
    n_bits = H.shape[1]
    parent = list(range(len(paths)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    pairs = []
    for s in range(0, len(H), 1024):
        blk = H[s:s + 1024]
        same = blk @ H.T + (1 - blk) @ (1 - H).T            # bits that agree
        dist = n_bits - same
        ii, jj = np.where(dist <= max_dist)
        for i, j in zip(ii + s, jj):
            if i < j:
                pairs.append((int(i), int(j), int(dist[i - s, j])))
                ri, rj = find(i), find(j)
                if ri != rj:
                    parent[ri] = rj
    groups = np.array([find(i) for i in range(len(paths))])
    return groups, pairs


def group_split(labels, groups, seed: int = 0, n_splits: int = 7):
    """Seven stratified folds: one test, one validation, five training. Near-duplicate groups stay whole, so the
    actual shares can differ from 1/7 each (the real split is reported from aptos_split.csv)."""
    from sklearn.model_selection import StratifiedGroupKFold
    sgk = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = np.zeros(len(labels), int)
    for k, (_, idx) in enumerate(sgk.split(np.zeros(len(labels)), labels, groups)):
        folds[idx] = k
    split = np.where(folds == 0, "test", np.where(folds == 1, "val", "train"))
    return split


# ============================================================ simple baseline features
def color_features(path) -> np.ndarray:
    """16-bin histograms per colour channel over the fundus (black border ignored) + mean/std."""
    a = np.asarray(Image.open(path).convert("RGB").resize((128, 128)), dtype=np.float32)
    m = a.sum(axis=2) > 30
    if m.sum() < 50:
        m = np.ones(a.shape[:2], bool)
    feats = []
    for ch in range(3):
        v = a[..., ch][m]
        h, _ = np.histogram(v, bins=16, range=(0, 255))
        feats += list(h / max(1, h.sum())) + [v.mean() / 255, v.std() / 255]
    return np.array(feats, np.float32)


# ============================================================ torch parts
def _torch():
    import torch
    return torch


def get_transforms(size: int, train: bool):
    from torchvision import transforms as T
    if train:
        return T.Compose([T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(25),
                          T.ColorJitter(0.2, 0.2, 0.1), T.ToTensor(), T.Normalize(MEAN, STD)])
    return T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])


def make_dataset(paths, labels, size, train, degrade_fn=None):
    """degrade_fn (optional) makes each photo worse in a fixed way before scoring; used only by the stress test."""
    from torch.utils.data import Dataset
    tf = get_transforms(size, train)

    class _DS(Dataset):
        def __len__(self):
            return len(paths)

        def __getitem__(self, i):
            img = Image.open(paths[i]).convert("RGB")
            if img.size != (size, size):
                img = img.resize((size, size))
            if degrade_fn is not None:
                img = degrade_fn(img)
            return tf(img), int(labels[i]) if labels is not None else -1
    return _DS()


def build_model(arch: str = "efficientnet_b0", pretrained: bool = True, n_classes: int = 5):
    import torch.nn as nn
    import torchvision
    if arch == "efficientnet_b0":
        m = torchvision.models.efficientnet_b0(weights="IMAGENET1K_V1" if pretrained else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)
    elif arch == "resnet50":
        m = torchvision.models.resnet50(weights="IMAGENET1K_V2" if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, n_classes)
    else:
        raise ValueError(arch)
    m.arch = arch
    return m


def _embed(model, x):
    """Penultimate-layer features (used by the linear-probe baseline and the quality gate)."""
    torch = _torch()
    if getattr(model, "arch", "") == "resnet50":
        f = torch.nn.Sequential(*list(model.children())[:-1])(x)
    else:
        f = model.avgpool(model.features(x))
    return torch.flatten(f, 1)


def predict(model, ds, batch_size=32, device=None, workers=2, with_embeddings=False):
    torch = _torch()
    from torch.utils.data import DataLoader
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.eval().to(device)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=workers)
    P, E = [], []
    with torch.no_grad():
        for x, _ in dl:
            x = x.to(device)
            with torch.autocast(device_type="cuda", enabled=device == "cuda"):
                if with_embeddings:
                    e = _embed(model, x)
                    logits = model.classifier(e) if hasattr(model, "classifier") else model.fc(e)
                    E.append(e.float().cpu().numpy())
                else:
                    logits = model(x)
            P.append(torch.softmax(logits.float(), 1).cpu().numpy())
    P = np.concatenate(P)
    return (P, np.concatenate(E)) if with_embeddings else P


def referable_prob(P: np.ndarray) -> np.ndarray:
    return P[:, REFERABLE_FROM:].sum(axis=1)


def train_model(model, train_ds, val_ds, val_grades, epochs=12, lr=3e-4, batch_size=32,
                device=None, workers=2, out_path="model_best.pt", log=print):
    """Train; keep the epoch with the best validation AUC for referable DR (validation only)."""
    torch = _torch()
    from torch.utils.data import DataLoader
    from sklearn.metrics import roc_auc_score
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=workers, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * max(1, len(dl)))
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")
    loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=0.05)
    yv = (np.asarray(val_grades) >= REFERABLE_FROM).astype(int)
    best, hist = -1.0, []
    for ep in range(epochs):
        model.train()
        t0, tot, n = time.time(), 0.0, 0
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", enabled=device == "cuda"):
                loss = loss_fn(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.item() * len(y)
            n += len(y)
        pv = referable_prob(predict(model, val_ds, batch_size, device, workers))
        auc = roc_auc_score(yv, pv) if len(set(yv)) > 1 else float("nan")
        hist.append(dict(epoch=ep + 1, train_loss=tot / max(1, n), val_auc=auc, seconds=time.time() - t0))
        log(f"epoch {ep + 1}/{epochs}: loss {tot / max(1, n):.3f}, validation AUC (referable) {auc:.3f}, {time.time() - t0:.0f}s")
        if auc > best:
            best = auc
            torch.save(model.state_dict(), out_path)
    model.load_state_dict(torch.load(out_path, map_location=device))
    return pd.DataFrame(hist)


# ============================================================ thresholds and metrics
def threshold_for_sensitivity(y_bin, p, target=0.90) -> float:
    """Highest threshold whose sensitivity on these (validation) cases is at least `target`."""
    y_bin, p = np.asarray(y_bin), np.asarray(p)
    pos = np.sort(p[y_bin == 1])[::-1]
    if len(pos) == 0:
        return 0.5
    k = int(np.ceil(target * len(pos))) - 1            # need at least k+1 positives above the line
    return float(pos[k])


def bootstrap_auc(y, p, n=1000, seed=0):
    from sklearn.metrics import roc_auc_score
    rng = np.random.default_rng(seed)
    y, p = np.asarray(y), np.asarray(p)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) > 1:
            vals.append(roc_auc_score(y[i], p[i]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    ph = k / n
    d = 1 + z * z / n
    c = (ph + z * z / (2 * n)) / d
    h = z * np.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / d
    return (float(c - h), float(c + h))


def binary_report(y_bin, p, thr, name="") -> dict:
    from sklearn.metrics import roc_auc_score
    y_bin, p = np.asarray(y_bin).astype(int), np.asarray(p)
    pred = p >= thr
    tp, fn = int((pred & (y_bin == 1)).sum()), int((~pred & (y_bin == 1)).sum())
    fp, tn = int((pred & (y_bin == 0)).sum()), int((~pred & (y_bin == 0)).sum())
    auc = roc_auc_score(y_bin, p) if len(set(y_bin)) > 1 else float("nan")
    return dict(model=name, n=len(y_bin), referable=int(y_bin.sum()), threshold=float(thr),
                AUC=auc, AUC_CI=bootstrap_auc(y_bin, p) if len(set(y_bin)) > 1 else (np.nan, np.nan),
                sensitivity=tp / max(1, tp + fn), sensitivity_CI=wilson(tp, tp + fn),
                specificity=tn / max(1, tn + fp), specificity_CI=wilson(tn, tn + fp),
                PPV=tp / max(1, tp + fp), NPV=tn / max(1, tn + fn),
                missed_FN=fn, false_alarms_FP=fp, referred_share=float(pred.mean()))


def grade_report(grades, P) -> dict:
    from sklearn.metrics import cohen_kappa_score, confusion_matrix
    g = np.asarray(grades).astype(int)
    pred = P.argmax(1)
    pr = referable_prob(P)
    per_grade = {GRADES[k]: dict(n=int((g == k).sum())) for k in range(5)}
    return dict(QWK=float(cohen_kappa_score(g, pred, weights="quadratic")),
                confusion=confusion_matrix(g, pred, labels=list(range(5))).tolist(),
                per_grade=per_grade, mean_referable_prob_by_grade=[float(pr[g == k].mean()) if (g == k).any() else None
                                                                   for k in range(5)])


def flagged_by_grade(grades, p, thr) -> pd.DataFrame:
    g = np.asarray(grades).astype(int)
    rows = []
    for k in range(5):
        m = g == k
        rows.append(dict(grade=f"{k} {GRADES[k]}", photos=int(m.sum()),
                         sent_to_specialist=int((p[m] >= thr).sum()),
                         share_sent=float((p[m] >= thr).mean()) if m.any() else np.nan))
    return pd.DataFrame(rows)


def fit_logreg(X, y, C=1.0):
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000, class_weight="balanced"))
    return m.fit(X, y)


# ============================================================ the clinic decision
def decide(p_ref, p_poor, ref_thr, poor_thr):
    """Refer whenever referable DR is suspected (a poor photo never blocks a referral).
    Otherwise a poor photo gets RETAKE instead of an all-clear."""
    p_ref, p_poor = np.asarray(p_ref), np.asarray(p_poor)
    return np.where(p_ref >= ref_thr, "REFER", np.where(p_poor >= poor_thr, "RETAKE", "RESCREEN"))


def workflow_report(grades, decisions) -> dict:
    g = np.asarray(grades).astype(int)
    d = np.asarray(decisions)
    ref = g >= REFERABLE_FROM
    return dict(photos=len(g),
                REFER=int((d == "REFER").sum()), RETAKE=int((d == "RETAKE").sum()), RESCREEN=int((d == "RESCREEN").sum()),
                referable_given_all_clear=int((ref & (d == "RESCREEN")).sum()),
                referable_caught_by_retake=int((ref & (d == "RETAKE")).sum()),
                sight_threatening_given_all_clear=int(((g >= 3) & (d == "RESCREEN")).sum()))


# ============================================================ stress test: worse photos
# Planned before any result was seen. Each change is applied to the 384 px photo, at two fixed strengths,
# to mimic what cheaper cameras, rushed photography or sending photos by phone do to image quality.
# It is a stress test of the frozen tool, not a validation on real low-cost cameras.
DEGRADATIONS = {
    #  name               what it imitates                                   mild   strong
    "blur":            ("out of focus or camera shake (Gaussian blur, px)",  1.5,   3.0),
    "low_resolution":  ("a cheaper, lower-resolution sensor (shrunk to px)", 192,   96),
    "underexposed":    ("too dark: poor lighting or a small pupil (brightness x)", 0.6, 0.35),
    "compressed":      ("sent over a messaging app (JPEG quality)",          30,    8),
}


def degrade(img: Image.Image, kind: str, level) -> Image.Image:
    """Make a photo worse in one fixed, repeatable way (no randomness)."""
    import io
    from PIL import ImageEnhance, ImageFilter
    img = img.convert("RGB")
    if kind == "clean":
        return img
    if kind == "blur":
        return img.filter(ImageFilter.GaussianBlur(float(level)))
    if kind == "low_resolution":
        w, h = img.size
        return img.resize((int(level), int(level)), Image.BILINEAR).resize((w, h), Image.BILINEAR)
    if kind == "underexposed":
        return ImageEnhance.Brightness(img).enhance(float(level))
    if kind == "compressed":
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=int(level))
        buf.seek(0)
        return Image.open(buf).convert("RGB")
    raise ValueError(kind)


def stress_conditions():
    """[(change, strength, level)], starting with the original photos."""
    out = [("clean", "none", None)]
    for kind, (_, mild, strong) in DEGRADATIONS.items():
        out += [(kind, "mild", mild), (kind, "strong", strong)]
    return out


# ============================================================ one photo, as the clinic app runs it
def triage_one(model, gate, img, size, ref_thr, poor_thr, tf=None):
    """Screen one photo exactly as the clinic app does: open, crop, resize, score, quality check, decide.
    img is a PIL image or a file path. Returns (decision, p_referable, p_poor, grade_probabilities, embedding)."""
    torch = _torch()
    model.eval()
    tf = tf or get_transforms(size, False)
    dev = next(model.parameters()).device
    x = tf(prep_image(img, size)).unsqueeze(0).to(dev)
    with torch.no_grad():
        e = _embed(model, x)
        head = model.classifier if hasattr(model, "classifier") else model.fc
        P = torch.softmax(head(e), 1).cpu().numpy()
    e = e.cpu().numpy()
    p_ref = float(referable_prob(P)[0])
    p_poor = float(gate.predict_proba(e)[0, 1]) if gate is not None else 0.0
    return str(decide([p_ref], [p_poor], ref_thr, poor_thr)[0]), p_ref, p_poor, P[0], e


def _cpu_name() -> str:
    try:
        with open("/proc/cpuinfo") as f:
            for line in f:
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
    except OSError:
        pass
    import platform
    return platform.processor() or platform.machine() or "unknown"


def cpu_benchmark(model, gate, paths, size, ref_thr, poor_thr, n=20, heatmaps=5) -> dict:
    """Time the clinic step on the CPU alone (no graphics card), starting from the original photo file.
    The first photo is a warm-up and is not timed."""
    import copy
    torch = _torch()
    m = copy.deepcopy(model).cpu().eval()
    paths = list(paths)[: n + 1]
    if len(paths) < 2:
        raise ValueError("need at least 2 photos to time")
    tf = get_transforms(size, False)
    triage_one(m, gate, paths[0], size, ref_thr, poor_thr, tf)
    times = []
    for p in paths[1:]:
        t = time.perf_counter()
        triage_one(m, gate, p, size, ref_thr, poor_thr, tf)
        times.append(time.perf_counter() - t)
    hm = []
    for p in paths[1: 1 + heatmaps]:
        t = time.perf_counter()
        gradcam(m, Image.open(p).convert("RGB"), size, device="cpu")
        hm.append(time.perf_counter() - t)
    return dict(photos_timed=len(times), median_seconds_per_photo=float(np.median(times)),
                slowest_seconds_per_photo=float(np.max(times)),
                heatmap_median_seconds=float(np.median(hm)) if hm else float("nan"),
                cpu=_cpu_name(), cpu_threads=int(torch.get_num_threads()),
                parameters=int(sum(q.numel() for q in m.parameters())))


# ============================================================ explanation
def gradcam(model, pil_img, size=384, device=None):
    """Heat map of where the model looked when scoring referable DR (Grad-CAM on the last conv block), drawn on the photo."""
    cam = gradcam_map(model, pil_img, size, device)
    cam_img = Image.fromarray(np.uint8(cam * 255)).resize((size, size), Image.BILINEAR)
    base = prep_image(pil_img, size)
    heat = np.asarray(cam_img, dtype=np.float32) / 255
    rgb = np.asarray(base, dtype=np.float32)
    overlay = rgb * (1 - 0.45 * heat[..., None]) + np.array([255, 40, 0]) * (0.45 * heat[..., None])
    return Image.fromarray(np.uint8(np.clip(overlay, 0, 255)))


def gradcam_map(model, pil_img, size=384, device=None):
    """The raw Grad-CAM map (last conv block, scaled to 0..1) for the referable score."""
    torch = _torch()
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.eval().to(device)
    x = get_transforms(size, False)(prep_image(pil_img, size)).unsqueeze(0).to(device)
    layer = model.features[-1] if hasattr(model, "features") else model.layer4
    store = {}
    h1 = layer.register_forward_hook(lambda m, i, o: store.__setitem__("a", o))
    h2 = layer.register_full_backward_hook(lambda m, gi, go: store.__setitem__("g", go[0]))
    try:
        logits = model(x)
        score = torch.logsumexp(logits[:, REFERABLE_FROM:], 1) - torch.logsumexp(logits, 1)
        model.zero_grad()
        score.sum().backward()
        w = store["g"].mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((w * store["a"]).sum(1))[0].detach().cpu().numpy()
    finally:
        h1.remove()
        h2.remove()
    cam = cam - cam.min()
    return cam / (cam.max() + 1e-8)


# ============================================================ report text
# Languages for the note. A language is offered to users only once a native speaker on the team has checked it,
# except the defaults for the Pakistan setting, which the submission checker refuses to ship unchecked.
# To add a language later (for example Uzbek, checked by Laziza): add an entry here with checked=False, add its
# sentence to every entry in REPORT (tests fail until all four are there), have the reviewer approve the wording,
# then set checked=True. Nothing else in the code changes.
LANGS = {
    "en": dict(name="English", dir="ltr", reviewer="whole team", checked=True),
    "ur": dict(name="اردو", dir="rtl", reviewer="Zainab", checked=False),   # set True after Zainab's check (task Z2)
    "hi": dict(name="हिन्दी", dir="ltr", reviewer="Varun", checked=False),   # hidden until Varun checks it
}
# Shown by default for the Pakistan clinic setting.
DEFAULT_LANGS = ("en", "ur")


def shown_langs():
    """Languages the app offers: every checked language, plus the defaults (which must be checked before submission)."""
    return [c for c, v in LANGS.items() if v["checked"] or c in DEFAULT_LANGS]


# Same meaning in every language: the action, why, and that the doctor confirms it.
REPORT = {
    "REFER": {
        "en": "Referable diabetic retinopathy suspected (moderate or worse). Recommended next step, for the doctor to confirm: refer to an eye specialist.",
        "hi": "मधुमेह से आँख के पर्दे (रेटिना) की मध्यम या उससे गंभीर बीमारी (डायबिटिक रेटिनोपैथी) की आशंका है। अनुशंसित अगला कदम, डॉक्टर की पुष्टि के बाद: नेत्र विशेषज्ञ के पास भेजें।",
        "ur": "ذیابیطس کی وجہ سے آنکھ کے پردے (ریٹینا) کی درمیانی یا زیادہ بیماری کا شبہ ہے۔ تجویز کردہ اگلا قدم، ڈاکٹر کی تصدیق کے بعد: آنکھوں کے ماہر کے پاس بھیجیں۔",
    },
    "RETAKE": {
        "en": "The photo is not clear enough to rule out disease. Recommended next step: retake the photo; if it is still unclear, refer to an eye specialist.",
        "hi": "तस्वीर इतनी साफ़ नहीं है कि बीमारी को नकारा जा सके। अनुशंसित अगला कदम: तस्वीर दोबारा लें; अगर फिर भी साफ़ न हो, तो नेत्र विशेषज्ञ के पास भेजें।",
        "ur": "تصویر اتنی واضح نہیں کہ بیماری کو رد کیا جا سکے۔ تجویز کردہ اگلا قدم: تصویر دوبارہ لیں؛ اگر پھر بھی واضح نہ ہو تو آنکھوں کے ماہر کے پاس بھیجیں۔",
    },
    "RESCREEN": {
        "en": "No referable diabetic retinopathy seen on this photo. Recommended next step: routine eye screening again in 12 months.",
        "hi": "इस तस्वीर में ऐसी डायबिटिक रेटिनोपैथी नहीं दिखी जिसके लिए विशेषज्ञ के पास भेजना ज़रूरी हो। अनुशंसित अगला कदम: 12 महीने बाद फिर से नियमित आँखों की जाँच।",
        "ur": "اس تصویر میں ریفر کرنے کے قابل ذیابیطس کی آنکھ کی بیماری نظر نہیں آئی۔ تجویز کردہ اگلا قدم: 12 ماہ بعد دوبارہ معمول کا معائنہ۔",
    },
    "DISCLAIMER": {
        "en": "Prototype for a student hackathon. Not medical advice and not a diagnosis.",
        "hi": "यह छात्रों के हैकाथॉन का एक प्रोटोटाइप है। यह चिकित्सीय सलाह या निदान नहीं है।",
        "ur": "یہ طلبہ کے ہیکاتھون کا تجرباتی نمونہ ہے۔ یہ طبی مشورہ یا تشخیص نہیں ہے۔",
    },
}


def slip_html(decision: str, langs=DEFAULT_LANGS) -> str:
    """The printable note for the doctor and patient, one block per language (Urdu right-to-left)."""
    blocks = []
    for code in langs:
        name, direction = LANGS[code]["name"], LANGS[code]["dir"]
        blocks.append(f"<div dir='{direction}' lang='{code}' style='margin:6px 0;line-height:1.5'>"
                      f"<b>{name}:</b> {REPORT[decision][code]}<br>"
                      f"<span style='opacity:0.7;font-size:0.9em'>{REPORT['DISCLAIMER'][code]}</span></div>")
    return "".join(blocks)


@dataclass
class Frozen:
    """Everything the clinic tool needs, fixed before the test sets were scored."""
    arch: str
    size: int
    ref_threshold: float
    poor_threshold: float
    target_sensitivity: float


In [ ]:
%%writefile run_dr.py
"""End-to-end experiment: APTOS 2019 (India) for training, DeepDRiD (China) as the untouched external test.

Used by the notebook; can also run from the command line:
  python run_dr.py --aptos-csv .../train.csv --aptos-dir .../train_images --ddr-root DeepDRiD --out results
"""
from __future__ import annotations

import argparse
import json
import os
import pickle
import time
from dataclasses import asdict, dataclass

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import drscreen as ds

INK, MUTED, GRID = "#1f2328", "#6a737d", "#e1e4e8"
C_MAIN, C_EXT, C_PROBE, C_COLOR = "#2f6fdb", "#7a3fd1", "#d9822b", "#8c959f"
C_GRAYBAR = "#6e7781"   # the "without" comparison: deliberately neutral


@dataclass
class RunConfig:
    aptos_csv: str
    aptos_dir: str
    ddr_root: str | None = None
    oiaddr_root: str | None = None       # v2 only: OIA-DDR (Li et al. 2019) photos added to training and validation
    out: str = "results"
    cache: str = "cache"
    size: int = 384
    arch: str = "efficientnet_b0"
    pretrained: bool = True
    epochs: int = 12
    lr: float = 3e-4
    batch_size: int = 32
    workers: int = 2
    target_sensitivity: float = 0.90     # fixed before any result was seen
    poor_threshold: float = 0.50         # balanced logistic regression, default cut
    dup_max_dist: int = 10
    seed: int = 0
    max_images: int | None = None        # for quick code checks only


# ------------------------------------------------------------------ data loading
def load_aptos(csv, img_dir):
    df = pd.read_csv(csv)
    ext = ".png"
    if df["id_code"].astype(str).str.contains(r"\.").any():
        ext = ""
    df["path"] = [os.path.join(img_dir, f"{i}{ext}") for i in df["id_code"].astype(str)]
    missing = (~df["path"].map(os.path.exists)).sum()
    if missing:
        alt = [p[:-4] + ".jpg" for p in df["path"]] if ext == ".png" else df["path"]
        df["path"] = [a if not os.path.exists(p) and os.path.exists(a) else p for p, a in zip(df["path"], alt)]
        missing = (~df["path"].map(os.path.exists)).sum()
    if missing:
        raise FileNotFoundError(f"{missing} APTOS images not found under {img_dir}")
    return df.rename(columns={"diagnosis": "grade"})[["id_code", "path", "grade"]]


IMG_EXT = (".jpg", ".jpeg", ".png", ".tif", ".tiff")


def load_oiaddr(root, log=print):
    """OIA-DDR grading photos (Li et al. 2019; 147 hospitals, 42 camera models; CC BY-NC-SA 4.0), from whatever folder
    layout the copy has. Labels are read from, in order: '<image> <grade>' text files, CSV files with an image column
    and a grade column, or class folders named 0 to 5. Grade 5 (ungradable) is dropped. Our own split is used later."""
    import re
    index = {}
    for d, _, fs in os.walk(root):
        for f in fs:
            if f.lower().endswith(IMG_EXT):
                index.setdefault(f, os.path.join(d, f))
                index.setdefault(os.path.splitext(f)[0], os.path.join(d, f))
    labels = {}
    for d, _, fs in os.walk(root):
        for f in fs:
            p = os.path.join(d, f)
            if f.lower().endswith(".txt"):
                for line in open(p, errors="ignore"):
                    m = re.match(r"^\s*(\S+\.(?:jpg|jpeg|png|tif|tiff))\s+(\d)\s*$", line, re.I)
                    if m:
                        labels.setdefault(os.path.basename(m.group(1)), int(m.group(2)))
            elif f.lower().endswith(".csv"):
                try:
                    c = pd.read_csv(p)
                except Exception:  # noqa: BLE001
                    continue
                gcol = next((k for k in c.columns if str(k).lower() in ("diagnosis", "label", "grade", "level", "dr_grade", "class")), None)
                icol = next((k for k in c.columns if c[k].astype(str).map(lambda v: v in index).mean() > 0.5), None)
                if gcol is not None and icol is not None:
                    for i, g in zip(c[icol].astype(str), c[gcol]):
                        if str(g).strip().isdigit():
                            labels.setdefault(os.path.basename(index[i]) if i in index else i, int(g))
    how = "label files"
    if not labels:          # class folders: .../0/x.jpg ... .../5/y.jpg
        how = "class folders"
        for k, p in index.items():
            parent = os.path.basename(os.path.dirname(p))
            if parent.isdigit() and os.path.splitext(k)[1]:
                labels.setdefault(k, int(parent))
    rows = [dict(id_code=k, path=index[k], grade=g) for k, g in labels.items() if k in index]
    df = pd.DataFrame(rows, columns=["id_code", "path", "grade"]).drop_duplicates("path")
    n_all = len(df)
    df = df[df.grade.between(0, 4)].reset_index(drop=True)
    log(f"OIA-DDR: {n_all} labelled photos found under {root} (from {how}); {n_all - len(df)} ungradable dropped; "
        f"grades {df.grade.value_counts().sort_index().to_dict()}")
    if len(df) < 100:
        raise FileNotFoundError(f"Fewer than 100 labelled OIA-DDR photos found under {root}: check the dataset is attached")
    return df


def load_deepdrid(root, part):
    """part: 'training' or 'validation'. Returns one row per photo with eye-level grade, quality and source."""
    folder = os.path.join(root, "regular_fundus_images", f"regular-fundus-{part}")
    lab = pd.read_csv(os.path.join(folder, f"regular-fundus-{part}.csv"))
    src_csv = os.path.join(folder, f"regular-fundus-source-{part}.csv")
    if os.path.exists(src_csv):
        lab = lab.merge(pd.read_csv(src_csv)[["image_id", "Source"]], on="image_id", how="left")
    else:
        lab["Source"] = "unknown"
    lab["grade"] = lab["left_eye_DR_Level"].fillna(lab["right_eye_DR_Level"]).astype(int)
    lab["poor_quality"] = (lab["Overall quality"] == 0).astype(int)
    lab["path"] = [os.path.join(folder, "Images", str(p), f"{i}.jpg") for p, i in zip(lab["patient_id"], lab["image_id"])]
    lab = lab[lab["path"].map(os.path.exists)].reset_index(drop=True)
    lab["part"] = part
    return lab[["patient_id", "image_id", "path", "grade", "poor_quality", "Source", "part"]]


# ------------------------------------------------------------------ figures
def _style(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.grid(color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def fig_roc(curves, path):
    from sklearn.metrics import roc_curve
    fig, ax = plt.subplots(figsize=(6.4, 5.6), dpi=150)
    _style(ax)
    for label, y, p, thr, color in curves:
        fpr, tpr, _ = roc_curve(y, p)
        ax.plot(fpr * 100, tpr * 100, color=color, lw=2, label=label)
        pred = p >= thr
        sens = (pred & (y == 1)).sum() / max(1, (y == 1).sum())
        fpr_pt = (pred & (y == 0)).sum() / max(1, (y == 0).sum())
        ax.plot([fpr_pt * 100], [sens * 100], "o", ms=7, color=color, markeredgecolor="white", markeredgewidth=1.5)
    ax.plot([0, 100], [0, 100], color=GRID, lw=1, ls="--")
    ax.set_xlabel("False alarms: healthy eyes sent to a specialist (%)", color=INK)
    ax.set_ylabel("Referable eyes caught (%)", color=INK)
    ax.set_title("Referable diabetic retinopathy: catches vs false alarms\n(dots = threshold frozen on validation)",
                 loc="left", fontsize=10.5, color=INK)
    ax.legend(frameon=False, fontsize=8, loc="lower left", bbox_to_anchor=(0, 1.015), ncol=2, borderaxespad=0)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def fig_gallery(rows, path, title, size=160):
    rows = rows[:8]
    if len(rows) == 0:
        return
    fig, axes = plt.subplots(1, len(rows), figsize=(2.0 * len(rows), 2.5), dpi=150)
    axes = np.atleast_1d(axes)
    for ax, r in zip(axes, rows):
        ax.imshow(Image.open(r["path"]).convert("RGB").resize((size, size)))
        ax.set_title(r["caption"], fontsize=7, color=INK)
        ax.axis("off")
    fig.suptitle(title, fontsize=10, color=INK, x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def stress_test(model, gate, ref_thr, poor_thr, sets, cfg, log=print) -> pd.DataFrame:
    """Score the frozen tool on deliberately worse copies of held-out photos. Nothing is re-tuned."""
    from functools import partial
    rows = []
    for set_name, d in sets:
        y = (d["grade"].to_numpy() >= ds.REFERABLE_FROM).astype(int)
        for kind, strength, level in ds.stress_conditions():
            fn = None if kind == "clean" else partial(ds.degrade, kind=kind, level=level)
            P, E = ds.predict(model, ds.make_dataset(list(d["cached"]), None, cfg.size, False, degrade_fn=fn),
                              cfg.batch_size, workers=cfg.workers, with_embeddings=True)
            p_ref = ds.referable_prob(P)
            p_poor = gate.predict_proba(E)[:, 1] if gate is not None else np.zeros(len(d))
            rep = ds.binary_report(y, p_ref, ref_thr)
            w = ds.workflow_report(d["grade"], ds.decide(p_ref, p_poor, ref_thr, poor_thr))
            w0 = ds.workflow_report(d["grade"], ds.decide(p_ref, np.zeros(len(d)), ref_thr, poor_thr))
            rows.append(dict(photos=set_name, change=kind, strength=strength, level=level, n=len(d), referable=int(y.sum()),
                             AUC=rep["AUC"], sensitivity=rep["sensitivity"], specificity=rep["specificity"],
                             retake_share=w["RETAKE"] / max(1, w["photos"]),
                             referable_all_clear_with_check=w["referable_given_all_clear"],
                             referable_all_clear_without_check=w0["referable_given_all_clear"],
                             grade3_4_all_clear_with_check=w["sight_threatening_given_all_clear"],
                             grade3_4_all_clear_without_check=w0["sight_threatening_given_all_clear"]))
            log(f"  stress {set_name:>8} {kind:>15} {strength:>6}: sensitivity {rep['sensitivity']:.3f}, "
                f"specificity {rep['specificity']:.3f}, referable all-clears {w['referable_given_all_clear']} "
                f"with check / {w0['referable_given_all_clear']} without")
    return pd.DataFrame(rows)


def fig_stress(st: pd.DataFrame, path):
    """Per photo set: sensitivity and specificity (left), and referable eyes given an all-clear
    with vs without the quality check (right). Horizontal bars so every condition label is readable."""
    sets = list(dict.fromkeys(st["photos"]))
    fig, axes = plt.subplots(len(sets), 2, figsize=(12, 4.2 * len(sets)), dpi=150, squeeze=False)
    for i, name in enumerate(sets):
        d = st[st.photos == name].reset_index(drop=True)
        labels = ["original" if c == "clean" else f"{c.replace('_', ' ')}, {s}" for c, s in zip(d.change, d.strength)]
        y = np.arange(len(d))[::-1]
        h = 0.36
        ax = axes[i, 0]
        _style(ax)
        ax.grid(axis="y", visible=False)
        ax.barh(y + h / 2, d.sensitivity * 100, height=h, color=C_MAIN, label="sensitivity (referable eyes caught)")
        ax.barh(y - h / 2, d.specificity * 100, height=h, color=C_PROBE, label="specificity (healthy eyes cleared)")
        ax.set_yticks(y, labels, fontsize=8)
        ax.set_xlim(0, 100)
        ax.set_xlabel("%", color=MUTED, fontsize=9)
        ax.set_title(f"{name}: accuracy at the frozen threshold", loc="left", fontsize=10, color=INK, pad=24)
        ax.legend(frameon=False, fontsize=8, loc="lower left", bbox_to_anchor=(0, 1.015), ncol=2, borderaxespad=0)
        ax = axes[i, 1]
        _style(ax)
        ax.grid(axis="y", visible=False)
        a, b = d.referable_all_clear_without_check, d.referable_all_clear_with_check
        ax.barh(y + h / 2, a, height=h, color=C_GRAYBAR, label="without quality check")
        ax.barh(y - h / 2, b, height=h, color=C_MAIN, label="with quality check")
        top = max(a.max(), b.max(), 1)
        for yy, va, vb in zip(y, a, b):
            ax.text(va + top * 0.01, yy + h / 2, str(int(va)), va="center", fontsize=7, color=MUTED)
            ax.text(vb + top * 0.01, yy - h / 2, str(int(vb)), va="center", fontsize=7, color=INK)
        ax.set_xlim(0, top * 1.12)
        ax.set_yticks(y, labels, fontsize=8)
        ax.set_xlabel(f"referable eyes given an all-clear (of {int(d.referable.iloc[0])})", color=MUTED, fontsize=9)
        ax.set_title(f"{name}: false all-clears", loc="left", fontsize=10, color=INK, pad=24)
        ax.legend(frameon=False, fontsize=8, loc="lower left", bbox_to_anchor=(0, 1.015), ncol=2, borderaxespad=0)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def fig_stress_examples(photo_path, size, path):
    conds = ds.stress_conditions()
    base = Image.open(photo_path).convert("RGB").resize((size, size))
    fig, axes = plt.subplots(1, len(conds), figsize=(1.7 * len(conds), 2.2), dpi=150)
    for ax, (kind, strength, level) in zip(axes, conds):
        ax.imshow(ds.degrade(base, kind, level))
        ax.set_title("original" if kind == "clean" else f"{kind.replace('_', ' ')}\n{strength}", fontsize=7, color=INK)
        ax.axis("off")
    fig.suptitle("The same photo under each stress-test change", fontsize=9, color=INK, x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


# ------------------------------------------------------------------ main experiment
def run(cfg: RunConfig, log=print) -> dict:
    import torch
    t0 = time.time()
    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)
    os.makedirs(cfg.out, exist_ok=True)
    out = {"config": asdict(cfg)}

    # 1. APTOS: cache, find near-duplicates, split by duplicate group
    ap = load_aptos(cfg.aptos_csv, cfg.aptos_dir)
    if cfg.max_images:
        ap = pd.concat([d.head(max(2, cfg.max_images // 5)) for _, d in ap.groupby("grade")]).reset_index(drop=True)
    ap["cached"] = ds.cache_images(list(ap["path"]), os.path.join(cfg.cache, f"aptos_{cfg.size}"), cfg.size, max(1, cfg.workers))
    groups, pairs = ds.duplicate_groups(list(ap["cached"]), cfg.dup_max_dist)
    ap["dup_group"] = groups
    conflicts = sum(1 for i, j, _ in pairs if ap.grade[i] != ap.grade[j])
    out["duplicates"] = dict(pairs=len(pairs), pairs_with_different_grades=conflicts,
                             images_in_groups=int((pd.Series(groups).value_counts() > 1).sum()))
    log(f"Near-duplicate pairs: {len(pairs)} ({conflicts} with different grades). Kept together when splitting.")
    ap["split"] = ds.group_split(ap["grade"].to_numpy(), groups, cfg.seed)
    ap.to_csv(os.path.join(cfg.out, "aptos_split.csv"), index=False)
    log(ap.groupby("split")["grade"].value_counts().unstack(fill_value=0).to_string())
    tr, va, te = (ap[ap.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
    ybin = lambda d: (d["grade"].to_numpy() >= ds.REFERABLE_FROM).astype(int)
    ox = None
    if cfg.oiaddr_root:      # v2: more cameras in training and validation; APTOS test and DeepDRiD unchanged
        ox = load_oiaddr(cfg.oiaddr_root, log)
        if cfg.max_images:
            ox = pd.concat([d.head(max(2, cfg.max_images // 5)) for _, d in ox.groupby("grade")]).reset_index(drop=True)
        ox["cached"] = ds.cache_images(list(ox["path"]), os.path.join(cfg.cache, f"oiaddr_{cfg.size}"), cfg.size, max(1, cfg.workers))
        g2, p2 = ds.duplicate_groups(list(ox["cached"]), cfg.dup_max_dist)
        ox["dup_group"] = g2
        ox["split"] = ds.group_split(ox["grade"].to_numpy(), g2, cfg.seed)
        ox.to_csv(os.path.join(cfg.out, "oiaddr_split.csv"), index=False)
        out["oiaddr"] = dict(photos=len(ox), near_duplicate_pairs=len(p2),
                             split=ox.split.value_counts().to_dict(), grades=ox.grade.value_counts().sort_index().to_dict())
        log("OIA-DDR split:\n" + ox.groupby("split")["grade"].value_counts().unstack(fill_value=0).to_string())
        tr = pd.concat([tr.assign(source="APTOS"), ox[ox.split == "train"].assign(source="OIA-DDR")], ignore_index=True)
        va = pd.concat([va.assign(source="APTOS"), ox[ox.split == "val"].assign(source="OIA-DDR")], ignore_index=True)
        log(f"Training photos: {len(tr)} (APTOS + OIA-DDR); validation photos: {len(va)} (both sources)")

    # 2. Baselines
    results = {}
    Fc = {s: np.stack([ds.color_features(p) for p in d["cached"]]) for s, d in (("train", tr), ("val", va), ("test", te))}
    m = ds.fit_logreg(Fc["train"], ybin(tr))
    pv, pt = m.predict_proba(Fc["val"])[:, 1], m.predict_proba(Fc["test"])[:, 1]
    thr = ds.threshold_for_sensitivity(ybin(va), pv, cfg.target_sensitivity)
    results["colour_baseline"] = ds.binary_report(ybin(te), pt, thr, "Colour histogram + logistic regression")
    curves_te = [("Colour baseline (APTOS test)", ybin(te), pt, thr, C_COLOR)]

    probe_model = ds.build_model(cfg.arch, cfg.pretrained)
    emb = {}
    for s, d in (("train", tr), ("val", va), ("test", te)):
        _, emb[s] = ds.predict(probe_model, ds.make_dataset(list(d["cached"]), d["grade"].to_numpy(), cfg.size, False),
                               cfg.batch_size, workers=cfg.workers, with_embeddings=True)
    m = ds.fit_logreg(emb["train"], ybin(tr), C=0.1)
    pv, pt = m.predict_proba(emb["val"])[:, 1], m.predict_proba(emb["test"])[:, 1]
    thr = ds.threshold_for_sensitivity(ybin(va), pv, cfg.target_sensitivity)
    results["linear_probe"] = ds.binary_report(ybin(te), pt, thr, "Pretrained features, not fine-tuned + logistic regression")
    curves_te.append(("Linear probe (APTOS test)", ybin(te), pt, thr, C_PROBE))
    del probe_model
    for k in ("colour_baseline", "linear_probe"):
        log(f"{results[k]['model']}: AUC {results[k]['AUC']:.3f}, sensitivity {results[k]['sensitivity']:.3f}, specificity {results[k]['specificity']:.3f}")

    # 3. Fine-tuned model; threshold frozen on validation
    model = ds.build_model(cfg.arch, cfg.pretrained)
    hist = ds.train_model(model, ds.make_dataset(list(tr["cached"]), tr["grade"].to_numpy(), cfg.size, True),
                          ds.make_dataset(list(va["cached"]), va["grade"].to_numpy(), cfg.size, False),
                          va["grade"].to_numpy(), cfg.epochs, cfg.lr, cfg.batch_size, workers=cfg.workers,
                          out_path=os.path.join(cfg.out, "model_best.pt"), log=log)
    hist.to_csv(os.path.join(cfg.out, "training_history.csv"), index=False)
    Pv, Ev = ds.predict(model, ds.make_dataset(list(va["cached"]), None, cfg.size, False), cfg.batch_size, workers=cfg.workers, with_embeddings=True)
    ref_thr = ds.threshold_for_sensitivity(ybin(va), ds.referable_prob(Pv), cfg.target_sensitivity)
    out["frozen"] = asdict(ds.Frozen(cfg.arch, cfg.size, ref_thr, cfg.poor_threshold, cfg.target_sensitivity))
    log(f"FROZEN referral threshold {ref_thr:.3f} (validation sensitivity target {cfg.target_sensitivity:.0%})")

    # 4. APTOS held-out test (scored once, after freezing)
    Pt = ds.predict(model, ds.make_dataset(list(te["cached"]), None, cfg.size, False), cfg.batch_size, workers=cfg.workers)
    pt = ds.referable_prob(Pt)
    results["fine_tuned"] = ds.binary_report(ybin(te), pt, ref_thr, f"Fine-tuned {cfg.arch}")
    out["aptos_grades"] = ds.grade_report(te["grade"], Pt)
    out["aptos_flagged_by_grade"] = ds.flagged_by_grade(te["grade"], pt, ref_thr)
    curves_te.append((f"Fine-tuned (APTOS test, India)", ybin(te), pt, ref_thr, C_MAIN))
    te = te.assign(p_referable=pt, pred_grade=Pt.argmax(1))
    te.to_csv(os.path.join(cfg.out, "aptos_test_predictions.csv"), index=False)
    log(f"APTOS test: AUC {results['fine_tuned']['AUC']:.3f}, sensitivity {results['fine_tuned']['sensitivity']:.3f}, "
        f"specificity {results['fine_tuned']['specificity']:.3f}, missed {results['fine_tuned']['missed_FN']}")

    if ox is not None:      # OIA-DDR held-out test: new photos from the same hospitals as its training part
        ot = ox[ox.split == "test"].reset_index(drop=True)
        Po = ds.predict(model, ds.make_dataset(list(ot["cached"]), None, cfg.size, False), cfg.batch_size, workers=cfg.workers)
        ot["p_referable"] = ds.referable_prob(Po)
        results["oiaddr_test"] = ds.binary_report(ybin(ot), ot["p_referable"].to_numpy(), ref_thr, "Fine-tuned, OIA-DDR test (China)")
        ot.drop(columns=["path", "cached"]).to_csv(os.path.join(cfg.out, "oiaddr_test_predictions.csv"), index=False)
        curves_te.append(("Fine-tuned (OIA-DDR test, China)", ybin(ot), ot["p_referable"].to_numpy(), ref_thr, C_PROBE))
        log(f"OIA-DDR test: AUC {results['oiaddr_test']['AUC']:.3f}, sensitivity {results['oiaddr_test']['sensitivity']:.3f}, "
            f"specificity {results['oiaddr_test']['specificity']:.3f}")

    # 5. External: DeepDRiD (never used to train the DR model or to set its threshold)
    if cfg.ddr_root:
        dd = pd.concat([load_deepdrid(cfg.ddr_root, "training"), load_deepdrid(cfg.ddr_root, "validation")], ignore_index=True)
        dd["cached"] = ds.cache_images(list(dd["path"]), os.path.join(cfg.cache, f"ddr_{cfg.size}"), cfg.size, max(1, cfg.workers))
        Pd, Ed = ds.predict(model, ds.make_dataset(list(dd["cached"]), None, cfg.size, False), cfg.batch_size,
                            workers=cfg.workers, with_embeddings=True)
        dd["p_referable"] = ds.referable_prob(Pd)
        dd["near_copy_in_training"] = False
        if ox is not None:      # DeepDRiD must stay unseen: look for near-copies among the OIA-DDR photos used
            used = ox[ox.split.isin(["train", "val"])]
            Hd = np.stack([ds.dhash(p) for p in dd["cached"]]).astype(np.float32)
            Ho = np.stack([ds.dhash(p) for p in used["cached"]]).astype(np.float32)
            dist = Hd.shape[1] - (Hd @ Ho.T + (1 - Hd) @ (1 - Ho).T)
            dd["near_copy_in_training"] = (dist <= cfg.dup_max_dist).any(1)
            out["deepdrid_near_copies_in_oiaddr"] = int(dd["near_copy_in_training"].sum())
            log(f"DeepDRiD photos with a near-copy among the OIA-DDR photos used: {out['deepdrid_near_copies_in_oiaddr']}")
        graded = dd["grade"] <= 4
        g = dd[graded]
        rep = ds.binary_report(ybin(g), g["p_referable"].to_numpy(), ref_thr, "Fine-tuned, external DeepDRiD (China)")
        rep["AUC_CI_by_patient"] = _patient_bootstrap_auc(ybin(g), g["p_referable"].to_numpy(), g["patient_id"].to_numpy())
        results["external_all"] = rep
        if dd["near_copy_in_training"].any():
            gc = g[~g["near_copy_in_training"]]
            results["external_no_near_copies"] = ds.binary_report(ybin(gc), gc["p_referable"].to_numpy(), ref_thr,
                                                                  "External DeepDRiD, near-copies of training photos removed")
        for q, name in ((0, "good"), (1, "poor")):
            gq = g[g.poor_quality == q]
            if len(gq) and len(set(ybin(gq))) > 1:
                results[f"external_{name}_photos"] = ds.binary_report(ybin(gq), gq["p_referable"].to_numpy(), ref_thr,
                                                                       f"External, {name}-quality photos")
        curves_te.append(("Fine-tuned (DeepDRiD, China, external)", ybin(g), g["p_referable"].to_numpy(), ref_thr, C_EXT))
        out["external_by_source"] = [dict(source=s, photos=len(d), **{k: v for k, v in ds.binary_report(
            ybin(d), d["p_referable"].to_numpy(), ref_thr).items() if k in ("sensitivity", "specificity", "missed_FN")})
            for s, d in g.groupby("Source") if len(d) >= 20 and len(set(ybin(d))) > 1]

        # 6. Quality gate: trained on DeepDRiD training photos' quality labels, tested on DeepDRiD validation
        trq, teq = dd.part.to_numpy() == "training", dd.part.to_numpy() == "validation"
        gate = ds.fit_logreg(Ed[trq], dd.loc[trq, "poor_quality"].to_numpy(), C=0.1)
        dd["p_poor"] = gate.predict_proba(Ed)[:, 1]
        from sklearn.metrics import roc_auc_score
        v = dd[teq]
        out["quality_gate"] = dict(AUC_detecting_poor_photos=float(roc_auc_score(v.poor_quality, v.p_poor)),
                                   flagged_share=float((v.p_poor >= cfg.poor_threshold).mean()),
                                   poor_photos_flagged=float((v.p_poor[v.poor_quality == 1] >= cfg.poor_threshold).mean()),
                                   good_photos_flagged=float((v.p_poor[v.poor_quality == 0] >= cfg.poor_threshold).mean()))
        vg = v[v.grade <= 4]
        with_gate = ds.decide(vg.p_referable, vg.p_poor, ref_thr, cfg.poor_threshold)
        no_gate = ds.decide(vg.p_referable, np.zeros(len(vg)), ref_thr, cfg.poor_threshold)
        out["workflow"] = dict(with_quality_gate=ds.workflow_report(vg.grade, with_gate),
                               without_quality_gate=ds.workflow_report(vg.grade, no_gate),
                               ungradable_photos=int((v.grade == 5).sum()),
                               ungradable_sent_to_retake_or_refer=int((ds.decide(v[v.grade == 5].p_referable, v[v.grade == 5].p_poor,
                                                                                ref_thr, cfg.poor_threshold) != "RESCREEN").sum()))
        dd["decision"] = ds.decide(dd.p_referable, dd.p_poor, ref_thr, cfg.poor_threshold)
        dd.to_csv(os.path.join(cfg.out, "deepdrid_predictions.csv"), index=False)
        with open(os.path.join(cfg.out, "quality_gate.pkl"), "wb") as f:
            pickle.dump(gate, f)
        log(f"External DeepDRiD: AUC {rep['AUC']:.3f}, sensitivity {rep['sensitivity']:.3f}, specificity {rep['specificity']:.3f}")
        log(f"Workflow on DeepDRiD validation: {out['workflow']}")

    # 7. Tables and figures
    summary = pd.DataFrame([{k: (f"{v:.3f}" if isinstance(v, float) else v) for k, v in r.items()
                             if k in ("model", "n", "referable", "AUC", "sensitivity", "specificity", "PPV", "NPV",
                                      "missed_FN", "false_alarms_FP", "referred_share")} for r in results.values()])
    summary.to_csv(os.path.join(cfg.out, "summary.csv"), index=False)
    fig_roc(curves_te, os.path.join(cfg.out, "roc.png"))
    misses = te[(te.grade >= ds.REFERABLE_FROM) & (te.p_referable < ref_thr)].sort_values("p_referable")
    fig_gallery([dict(path=r.cached, caption=f"grade {r.grade} ({ds.GRADES[r.grade]})\nscore {r.p_referable:.2f}")
                 for r in misses.itertuples()], os.path.join(cfg.out, "missed_cases.png"),
                "Referable eyes the model missed (APTOS test) — look for why")
    if cfg.ddr_root:   # external misses: DeepDRiD photos (CC BY-SA 4.0), safe to show publicly with attribution
        mx = dd[(dd.grade >= ds.REFERABLE_FROM) & (dd.decision == "RESCREEN")].sort_values("p_referable")
        fig_gallery([dict(path=r.cached, caption=f"grade {r.grade} ({ds.GRADES[r.grade]})\nscore {r.p_referable:.2f}, "
                                                 f"{'poor' if r.poor_quality else 'good'} photo")
                     for r in mx.itertuples()], os.path.join(cfg.out, "missed_external.png"),
                    "Referable eyes given an all-clear on external DeepDRiD photos (CC BY-SA 4.0)")
    fps = te[(te.grade < ds.REFERABLE_FROM) & (te.p_referable >= ref_thr)].sort_values("p_referable", ascending=False)
    fig_gallery([dict(path=r.cached, caption=f"grade {r.grade} ({ds.GRADES[r.grade]})\nscore {r.p_referable:.2f}")
                 for r in fps.itertuples()], os.path.join(cfg.out, "false_alarms.png"),
                "Eyes sent to a specialist that did not need it (APTOS test)")
    # 8. Speed on the CPU alone (no graphics card), from the original photo file to a decision.
    #    Optional: if it fails, the run still finishes and the write-up says speed was not measured.
    try:
        if cfg.ddr_root:
            bench_paths, bench_gate = list(dd.loc[dd.part == "validation", "path"]), gate
        else:
            bench_paths, bench_gate = list(te["path"]), None
        cs = ds.cpu_benchmark(model, bench_gate, bench_paths, cfg.size, ref_thr, cfg.poor_threshold)
        cs["model_file_MB"] = os.path.getsize(os.path.join(cfg.out, "model_best.pt")) / 1e6
        out["cpu_speed"] = cs
        log(f"CPU only: {cs['median_seconds_per_photo']:.2f} s per photo (median of {cs['photos_timed']}), "
            f"heat map {cs['heatmap_median_seconds']:.2f} s, {cs['cpu_threads']} threads, {cs['cpu']}, "
            f"model file {cs['model_file_MB']:.1f} MB")
    except Exception as e:  # never lose a finished run over a speed test
        out["cpu_speed"] = dict(error=repr(e))
        log(f"CPU speed test skipped: {e!r}")

    # 9. Stress test: the frozen tool on deliberately worse copies of held-out photos (planned before results).
    #    Optional: if it fails, the run still finishes and the write-up says it was not run.
    try:
        sets = [("APTOS test", te)]
        if cfg.ddr_root:
            sets.append(("DeepDRiD validation", dd[(dd.part == "validation") & (dd.grade <= 4)].reset_index(drop=True)))
        log("Stress test (worse photos):")
        st = stress_test(model, gate if cfg.ddr_root else None, ref_thr, cfg.poor_threshold, sets, cfg, log)
        st.to_csv(os.path.join(cfg.out, "stress_test.csv"), index=False)
        out["stress_test"] = st.to_dict("records")
        fig_stress(st, os.path.join(cfg.out, "stress_test.png"))
        ex = sets[-1][1]
        ex = ex[ex.grade >= ds.REFERABLE_FROM] if (ex.grade >= ds.REFERABLE_FROM).any() else ex
        fig_stress_examples(ex["cached"].iloc[0], cfg.size, os.path.join(cfg.out, "stress_examples.png"))
    except Exception as e:  # never lose a finished run over the stress test
        out["stress_test_error"] = repr(e)
        log(f"Stress test skipped: {e!r}")

    out["results"] = results
    out["seconds"] = time.time() - t0
    with open(os.path.join(cfg.out, "results.json"), "w") as f:
        json.dump(out, f, indent=2, default=lambda o: o.to_dict("records") if isinstance(o, pd.DataFrame) else (o.tolist() if hasattr(o, "tolist") else str(o)))
    log(f"Done in {out['seconds'] / 60:.1f} min. Results in {cfg.out}/")
    out["summary"] = summary
    out["model"] = model
    return out


def _patient_bootstrap_auc(y, p, patients, n=1000, seed=0):
    """AUC interval resampling whole patients (each has up to 4 photos), so it is not falsely narrow."""
    from sklearn.metrics import roc_auc_score
    rng = np.random.default_rng(seed)
    pats = np.unique(patients)
    idx = {q: np.where(patients == q)[0] for q in pats}
    vals = []
    for _ in range(n):
        pick = np.concatenate([idx[q] for q in rng.choice(pats, len(pats))])
        if len(set(y[pick])) > 1:
            vals.append(roc_auc_score(y[pick], p[pick]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


if __name__ == "__main__":
    a = argparse.ArgumentParser()
    a.add_argument("--aptos-csv", required=True)
    a.add_argument("--aptos-dir", required=True)
    a.add_argument("--ddr-root", default=None, help="DeepDRiD (external test)")
    a.add_argument("--oiaddr-root", default=None, help="v2 only: OIA-DDR photos added to training and validation")
    a.add_argument("--out", default="results")
    a.add_argument("--epochs", type=int, default=12)
    a.add_argument("--size", type=int, default=384)
    a.add_argument("--no-pretrained", action="store_true")
    a.add_argument("--max-images", type=int, default=None)
    a.add_argument("--workers", type=int, default=2)
    x = a.parse_args()
    r = run(RunConfig(aptos_csv=x.aptos_csv, aptos_dir=x.aptos_dir, ddr_root=x.ddr_root, oiaddr_root=x.oiaddr_root,
                      out=x.out, size=x.size, epochs=x.epochs, pretrained=not x.no_pretrained,
                      max_images=x.max_images, workers=x.workers))
    print(r["summary"].to_string(index=False))


In [ ]:
import os, sys, glob, subprocess, hashlib
import torch, torchvision
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (this will be slow)')
print('code fingerprint (sha256, first 12):', {f: hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12] for f in ['drscreen.py', 'run_dr.py']})

## 2. Data

In [ ]:
APTOS = os.environ.get('APTOS_DIR') or next((c for c in ['/kaggle/input/aptos2019-blindness-detection',
        '/kaggle/input/competitions/aptos2019-blindness-detection', 'aptos'] if os.path.exists(os.path.join(c, 'train.csv'))), None)
assert APTOS, 'Attach the APTOS 2019 Blindness Detection competition data (Add Input)'
print('APTOS:', APTOS, '| photos:', len(glob.glob(os.path.join(APTOS, 'train_images', '*'))))

DDR = os.environ.get('DDR_ROOT', 'DeepDRiD')      # DeepDRiD: the external test, never trained on
if not os.path.exists(os.path.join(DDR, 'regular_fundus_images', 'regular-fundus-validation', 'regular-fundus-validation.csv')):
    os.system(f'git clone -q --depth 1 --filter=blob:none --sparse https://github.com/deepdrdoc/DeepDRiD.git {DDR}')
    os.system(f'cd {DDR} && git sparse-checkout set regular_fundus_images/regular-fundus-training regular_fundus_images/regular-fundus-validation')
print('DeepDRiD photos:', len(glob.glob(os.path.join(DDR, 'regular_fundus_images', '*', 'Images', '*', '*.jpg'))))

def find_oiaddr():
    if os.environ.get('OIADDR_ROOT'): return os.environ['OIADDR_ROOT']
    for d, subdirs, _ in os.walk('/kaggle/input'):
        name = os.path.basename(d).lower()
        if 'ddr' in name and 'aptos' not in d.lower() and 'deepdrid' not in d.lower():
            return d
        if d.count(os.sep) > 5: subdirs[:] = []
OIADDR = find_oiaddr()
assert OIADDR, 'Attach the ddrdataset (mariaherrerot) with Add Input, or set OIADDR_ROOT'
print('OIA-DDR folder:', OIADDR)
for d, subdirs, fs in os.walk(OIADDR):     # a short map of the folder, in case the layout needs checking
    depth = d[len(OIADDR):].count(os.sep)
    if depth <= 3:
        imgs = sum(f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif')) for f in fs)
        other = [f for f in fs if not f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif'))][:5]
        print('  ' * depth + (os.path.basename(d) or d) + f'/  images: {imgs}  other files: {other}')
    if depth >= 3: subdirs[:] = []
import run_dr
ox = run_dr.load_oiaddr(OIADDR)
assert len(ox) > int(os.environ.get('MIN_OIADDR', 5000)), f'Only {len(ox)} OIA-DDR photos with grades found: send the folder map above to Claude before running'

## 3. Quick check (about 5 minutes)
Everything on a few hundred photos for 2 epochs, so a broken step shows up early. Its numbers mean nothing.

In [ ]:
WORKERS = min(4, os.cpu_count() or 2)
TESTQ = os.environ.get('ROSHNI_TEST') == '1'
quick = run_dr.run(run_dr.RunConfig(f'{APTOS}/train.csv', f'{APTOS}/train_images', DDR, oiaddr_root=OIADDR,
                                    out='quick_check_v2', cache='cache', epochs=1 if TESTQ else 2, workers=WORKERS, max_images=150 if TESTQ else 300,
                                    size=128 if TESTQ else 384, pretrained=not TESTQ))
quick['summary']

## 4. Full v2 run (settings as pre-registered)

In [ ]:
TEST = os.environ.get('ROSHNI_TEST') == '1'     # small runs to check the code only; never for results
cfg = run_dr.RunConfig(f'{APTOS}/train.csv', f'{APTOS}/train_images', DDR, oiaddr_root=OIADDR,
                       out='results_v2', cache='cache', workers=WORKERS)
if TEST: cfg.epochs, cfg.size, cfg.pretrained, cfg.max_images = 1, 128, False, 150
assert TEST or (cfg.arch, cfg.size, cfg.epochs, cfg.lr, cfg.batch_size, cfg.seed, cfg.target_sensitivity, cfg.poor_threshold) == \
       ('efficientnet_b0', 384, 12, 3e-4, 32, 0, 0.90, 0.50), 'settings differ from the pre-registration'
out = run_dr.run(cfg)
out['summary']

## 5. The pre-registered decision

In [ ]:
import json
r = json.load(open('results_v2/results.json'))
ext, wf, ap = r['results']['external_all'], r['workflow']['with_quality_gate'], r['results']['fine_tuned']
checks = {'DeepDRiD sensitivity >= 66.1%': ext['sensitivity'] >= 0.661,
          'DeepDRiD specificity >= 85.0%': ext['specificity'] >= 0.850,
          'APTOS test AUC >= 0.950': ap['AUC'] >= 0.950,
          'false all-clears (400-photo workflow, with check) <= 46': wf['referable_given_all_clear'] <= 46,
          'fewer than 16 DeepDRiD photos with a near-copy in OIA-DDR training/validation': r.get('deepdrid_near_copies_in_oiaddr', 0) < 16}
for k, v in checks.items(): print(('PASS ' if v else 'FAIL ') + k)
print(f"DeepDRiD: sensitivity {ext['sensitivity']:.1%}, specificity {ext['specificity']:.1%}, AUC {ext['AUC']:.3f}; "
      f"false all-clears {wf['referable_given_all_clear']}; APTOS test AUC {ap['AUC']:.3f}")
print('Near-copies of training photos among DeepDRiD photos:', r.get('deepdrid_near_copies_in_oiaddr'))
print('v2 becomes the demo model' if all(checks.values()) else 'v1 stays the demo model; v2 is reported in full')

## 6. Download
Download `roshni_v2_outputs.zip` from this version's Output tab. It holds `results_v2/` (no photos are copied into it; the gallery figures of APTOS photos stay team-only).

In [ ]:
import shutil
shutil.make_archive('roshni_v2_outputs', 'zip', '.', 'results_v2')
print(os.path.getsize('roshni_v2_outputs.zip') / 1e6, 'MB')